# Stack with Ridge meta-learner
Same base models and OOF setup as 98 (LightGBM + LSTM + Ridge + EWM). Meta-learner: **Ridge(positive=True)** with **StandardScaler** on meta features (4 base preds + disagreement + month_sin, month_cos, vix_velocity, rolling_vol + symbol one-hot). For comparison with 98-xgb-lstm-stack-pool.

In [23]:
import sys
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")

from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import Ridge
import lightgbm as lgb

REPO_ROOT = Path.cwd().parent.parent
BACKEND_DIR = REPO_ROOT / "backend"
sys.path.insert(0, str(BACKEND_DIR))
sys.path.insert(0, str(Path.cwd()))

from _pool_common import (
    load_pool_data,
    build_pooled_train_stack,
    compute_metrics_averaged_over_windows,
    metrics_to_parquet,
    fetch_cnn_fear_greed_index,
    TEST_SIZE,
    FORECAST_HORIZON,
    ROLLING_STEP,
    MIN_TRAIN_STACK,
    ARTIFACTS_DIR,
    TICKERS,
)

import json
# Load best params from 05/06 random search if available (run 05 and 06 first to generate artifacts)
_lgb_path = ARTIFACTS_DIR / "best_lgb_params.json"
if _lgb_path.exists():
    with open(_lgb_path) as f:
        LGB_PARAMS = json.load(f)
    LGB_PARAMS.setdefault("random_state", 42)
    LGB_PARAMS.setdefault("verbosity", -1)
else:
    LGB_PARAMS = dict(n_estimators=100, max_depth=4, learning_rate=0.01, random_state=42, verbosity=-1)
_ridge_path = ARTIFACTS_DIR / "best_ridge_params.json"
if _ridge_path.exists():
    with open(_ridge_path) as f:
        _ridge = json.load(f)
    RIDGE_ALPHA = _ridge["alpha"]
else:
    RIDGE_ALPHA = 1.0
EWM_SPAN = 20
ROLL_VOL_WINDOW = 21  # rolling std of returns for meta rolling_vol feature

# Feature config: LSTM vs LightGBM use different feature sets (LGB: vix_sma_5, fear_greed_change)
LAG_RETURNS = 5
RSI_PERIOD = 14
MACD_FAST, MACD_SLOW, MACD_SIGNAL = 12, 26, 9
SEQ_LEN = 30  # LSTM sequence length
LSTM_EPOCHS_OOF = 40  # LSTM epochs during OOF fold training
LSTM_EPOCHS_FINAL = 100  # max LSTM epochs for final base fit (early stop if converged)
RIDGE_META_ALPHA = 1.0  # Ridge meta-learner (positive=True); meta features scaled before fit
N_FOLDS = 5  # OOF: TimeSeriesSplit so base models only see past data (no chronological leakage)

In [24]:
def _rsi(series: pd.Series, period: int) -> pd.Series:
    """RSI = 100 - 100/(1 + RS), RS = avg gain / avg loss (Wilder)."""
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = (-delta).clip(lower=0)
    avg_gain = gain.ewm(alpha=1 / period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1 / period, adjust=False).mean()
    rs = avg_gain / np.where(avg_loss != 0, avg_loss, 1e-10)
    return 100 - (100 / (1 + rs))


def build_feature_df(grp: pd.DataFrame):
    """Build features: LSTM = return lags, volume lag, RSI, MACD; LGB = VIX 5-day SMA, cyclical month, fear_greed_change (lag1 - lag5). Target = next 21 returns."""
    df = grp.sort_values("timestamp").copy()
    df["close"] = df["close"].astype(float)
    df["return"] = df["close"].pct_change()
    for i in range(1, LAG_RETURNS + 1):
        df[f"ret_lag_{i}"] = df["return"].shift(i)
    if "volume" in df.columns:
        df["volume_lag_1"] = df["volume"].astype(float).shift(1)
    else:
        df["volume_lag_1"] = np.nan
    df["rsi"] = _rsi(df["close"], RSI_PERIOD)
    ema_fast = df["close"].ewm(span=MACD_FAST, adjust=False).mean()
    ema_slow = df["close"].ewm(span=MACD_SLOW, adjust=False).mean()
    df["macd_line"] = ema_fast - ema_slow
    df["macd_signal"] = df["macd_line"].ewm(span=MACD_SIGNAL, adjust=False).mean()
    if "vix" in df.columns:
        vix = df["vix"].astype(np.float64)
        df["vix_sma_5"] = vix.shift(1).rolling(5).mean()
        df["vix_velocity"] = vix.diff(1)
        df["vix_momentum"] = vix - df["vix_sma_5"]
    else:
        df["vix_sma_5"] = np.nan
        df["vix_velocity"] = np.nan
        df["vix_momentum"] = np.nan
    df["month"] = pd.to_datetime(df["timestamp"]).dt.month
    df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
    if "fear_greed" not in df.columns:
        df["fear_greed"] = 50.0
    else:
        df["fear_greed"] = df["fear_greed"].fillna(50.0)
    df["fear_greed_lag_1"] = df["fear_greed"].shift(1)
    df["fear_greed_lag_5"] = df["fear_greed"].shift(5)
    df["fear_greed_change"] = df["fear_greed_lag_1"] - df["fear_greed_lag_5"]
    df["rolling_vol"] = df["return"].rolling(ROLL_VOL_WINDOW, min_periods=1).std().fillna(0).astype(np.float32)
    for h in range(1, FORECAST_HORIZON + 1):
        df[f"target_{h}"] = df["return"].shift(-h)
    # LSTM: returns lag, volume lag, RSI, MACD
    feature_cols_lstm = [f"ret_lag_{i}" for i in range(1, LAG_RETURNS + 1)] + [
        "volume_lag_1", "rsi", "macd_line", "macd_signal"
    ]
    # LightGBM/Ridge base: VIX 5-day SMA, vix_velocity, vix_momentum, cyclical month, fear_greed change
    feature_cols_xgb = ["vix_sma_5", "vix_velocity", "vix_momentum", "month_sin", "month_cos", "fear_greed_change"]
    target_cols = [f"target_{h}" for h in range(1, FORECAST_HORIZON + 1)]
    base_cols = ["timestamp", "close", "return", "rolling_vol"] + feature_cols_lstm + feature_cols_xgb + target_cols
    out = df[[c for c in base_cols if c in df.columns]].copy()
    return out.dropna(), feature_cols_lstm, feature_cols_xgb, target_cols

def build_sequences(X: np.ndarray, y: np.ndarray, seq_len: int):
    """Build (n_seq, seq_len, n_feat) and (n_seq, horizon) for LSTM."""
    n = len(X)
    if n < seq_len + 1:
        return None, None
    X_seq, y_seq = [], []
    for i in range(seq_len, n):
        X_seq.append(X[i - seq_len : i])
        y_seq.append(y[i])
    return np.array(X_seq), np.array(y_seq)

In [25]:
try:
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import LSTM, Dense
    from tensorflow.keras.callbacks import EarlyStopping
    HAS_LSTM = True
except Exception:
    HAS_LSTM = False

def train_lstm(X_seq: np.ndarray, y_seq: np.ndarray, horizon: int, epochs=20, use_early_stopping=False):
    if not HAS_LSTM or X_seq is None or len(X_seq) < 10:
        return None
    n_feat = X_seq.shape[2]
    model = Sequential([
        LSTM(32, activation="tanh", input_shape=(X_seq.shape[1], n_feat)),
        Dense(horizon),
    ])
    model.compile(optimizer="adam", loss="mse")
    callbacks = []
    if use_early_stopping and HAS_LSTM:
        callbacks.append(EarlyStopping(monitor="loss", patience=5, restore_best_weights=True))
    model.fit(X_seq, y_seq, epochs=epochs, callbacks=callbacks, verbose=0)
    return model

In [26]:
def train_global_stack_ridge(stacked: pd.DataFrame, horizon: int):
    """OOF: Train base models per TimeSeriesSplit fold; stack OOF predictions to train Ridge meta-learner (scaled); then retrain base models on full training set. Returns dict for predict_stack_ridge_global."""
    pooled = build_pooled_train_stack(stacked, TEST_SIZE, MIN_TRAIN_STACK)
    if pooled.empty:
        return None
    feat_dfs = []
    for sym in pooled["symbol"].unique():
        grp = pooled[pooled["symbol"] == sym].copy()
        try:
            feat_df, feature_cols_lstm, feature_cols_xgb, target_cols = build_feature_df(grp)
        except Exception:
            continue
        if len(feat_df) < MIN_TRAIN_STACK + horizon:
            continue
        feat_df["_symbol_"] = sym
        feat_dfs.append(feat_df)
    if not feat_dfs:
        return None
    pooled_feat = pd.concat(feat_dfs, ignore_index=True)
    pooled_feat = pooled_feat.sort_values("timestamp").reset_index(drop=True)
    pooled_feat["_sym_idx"] = pooled_feat.groupby("_symbol_").cumcount()
    n_per_sym = pooled_feat.groupby("_symbol_").size()
    n_total = len(pooled_feat)
    if n_total < 100:
        return None
    sym_to_global = {sym: np.where(pooled_feat["_symbol_"] == sym)[0] for sym in pooled_feat["_symbol_"].unique()}
    tscv = TimeSeriesSplit(n_splits=N_FOLDS)
    oof_lgb = np.full((n_total, horizon), np.nan, dtype=np.float32)
    oof_lstm = np.full((n_total, horizon), np.nan, dtype=np.float32)
    oof_ridge = np.full((n_total, horizon), np.nan, dtype=np.float32)
    oof_ewm = np.full((n_total, horizon), np.nan, dtype=np.float32)
    feature_cols_ridge = feature_cols_lstm + feature_cols_xgb
    for train_idx, val_idx in tscv.split(pooled_feat):
        base_train_feat = pooled_feat.iloc[train_idx].copy()
        y_base = base_train_feat[target_cols].values.astype(np.float32)
        scaler_xgb_f, scaler_lstm_f, scaler_ridge_f = {}, {}, {}
        for sym in base_train_feat["_symbol_"].unique():
            m = base_train_feat["_symbol_"] == sym
            scaler_xgb_f[sym] = StandardScaler().fit(base_train_feat.loc[m, feature_cols_xgb].values.astype(np.float32))
            scaler_lstm_f[sym] = StandardScaler().fit(base_train_feat.loc[m, feature_cols_lstm].values.astype(np.float32))
            scaler_ridge_f[sym] = StandardScaler().fit(base_train_feat.loc[m, feature_cols_ridge].values.astype(np.float64))
        fallback_xgb = StandardScaler().fit(base_train_feat[feature_cols_xgb].values.astype(np.float32))
        fallback_lstm = StandardScaler().fit(base_train_feat[feature_cols_lstm].values.astype(np.float32))
        fallback_ridge = StandardScaler().fit(base_train_feat[feature_cols_ridge].values.astype(np.float64))
        X_xgb_base_s = np.empty((len(base_train_feat), len(feature_cols_xgb)), dtype=np.float32)
        X_lstm_base_s = np.empty((len(base_train_feat), len(feature_cols_lstm)), dtype=np.float32)
        for sym in base_train_feat["_symbol_"].unique():
            m = base_train_feat["_symbol_"] == sym
            X_xgb_base_s[m] = scaler_xgb_f[sym].transform(base_train_feat.loc[m, feature_cols_xgb].values.astype(np.float32))
            X_lstm_base_s[m] = scaler_lstm_f[sym].transform(base_train_feat.loc[m, feature_cols_lstm].values.astype(np.float32))
        lgb_multi = MultiOutputRegressor(lgb.LGBMRegressor(**LGB_PARAMS))
        lgb_multi.fit(X_xgb_base_s, y_base)
        X_seq_list, y_seq_list = [], []
        for sym in base_train_feat["_symbol_"].unique():
            sym_mask = base_train_feat["_symbol_"] == sym
            X_s = X_lstm_base_s[sym_mask]
            y_s = y_base[sym_mask]
            X_seq, y_seq = build_sequences(X_s, y_s, SEQ_LEN)
            if X_seq is not None and len(X_seq) >= 10:
                X_seq_list.append(X_seq)
                y_seq_list.append(y_seq)
        lstm_model = train_lstm(np.concatenate(X_seq_list, axis=0), np.concatenate(y_seq_list, axis=0), horizon, epochs=LSTM_EPOCHS_OOF) if X_seq_list else None
        X_ridge_base_s = np.empty((len(base_train_feat), len(feature_cols_ridge)), dtype=np.float64)
        for sym in base_train_feat["_symbol_"].unique():
            m = base_train_feat["_symbol_"] == sym
            X_ridge_base_s[m] = scaler_ridge_f[sym].transform(base_train_feat.loc[m, feature_cols_ridge].values.astype(np.float64))
        ridge_multi = MultiOutputRegressor(Ridge(alpha=RIDGE_ALPHA, random_state=42))
        ridge_multi.fit(X_ridge_base_s, y_base.astype(np.float64))
        X_xgb_full_s = np.empty((len(pooled_feat), len(feature_cols_xgb)), dtype=np.float32)
        X_lstm_full_s = np.empty((len(pooled_feat), len(feature_cols_lstm)), dtype=np.float32)
        X_ridge_full_s = np.empty((len(pooled_feat), len(feature_cols_ridge)), dtype=np.float64)
        for sym in pooled_feat["_symbol_"].unique():
            m = pooled_feat["_symbol_"] == sym
            sx = scaler_xgb_f.get(sym, fallback_xgb)
            sl = scaler_lstm_f.get(sym, fallback_lstm)
            sr = scaler_ridge_f.get(sym, fallback_ridge)
            X_xgb_full_s[m] = sx.transform(pooled_feat.loc[m, feature_cols_xgb].values.astype(np.float32))
            X_lstm_full_s[m] = sl.transform(pooled_feat.loc[m, feature_cols_lstm].values.astype(np.float32))
            X_ridge_full_s[m] = sr.transform(pooled_feat.loc[m, feature_cols_ridge].values.astype(np.float64))
        valid_val_list = []
        for i in val_idx:
            sym = pooled_feat.iloc[i]["_symbol_"]
            sym_idx = int(pooled_feat.iloc[i]["_sym_idx"])
            n_sym = int(n_per_sym[sym])
            if sym_idx < SEQ_LEN or sym_idx >= n_sym - horizon:
                continue
            valid_val_list.append(i)
        if valid_val_list:
            valid_idx = np.array(valid_val_list)
            oof_lgb[valid_idx] = lgb_multi.predict(X_xgb_full_s[valid_idx])
            seq_list = []
            for i in valid_idx:
                sym = pooled_feat.iloc[i]["_symbol_"]
                sym_idx = int(pooled_feat.iloc[i]["_sym_idx"])
                gidx = sym_to_global[sym]
                seq_idx = gidx[sym_idx - SEQ_LEN : sym_idx]
                seq_list.append(X_lstm_full_s[seq_idx])
            seq_batch = np.array(seq_list)
            if lstm_model is not None:
                oof_lstm[valid_idx] = lstm_model.predict(seq_batch, verbose=0)
            else:
                oof_lstm[valid_idx] = oof_lgb[valid_idx]
            oof_ridge[valid_idx] = ridge_multi.predict(X_ridge_full_s[valid_idx])
            for i in valid_idx:
                sym = pooled_feat.iloc[i]["_symbol_"]
                sym_idx = int(pooled_feat.iloc[i]["_sym_idx"])
                mask = (pooled_feat["_symbol_"] == sym) & (pooled_feat["_sym_idx"] <= sym_idx)
                r = pooled_feat.loc[mask, "return"].values.astype(np.float64)
                if len(r) >= EWM_SPAN:
                    ewm_val = float(pd.Series(r).ewm(span=EWM_SPAN).mean().iloc[-1])
                    oof_ewm[i, :] = ewm_val
    y = pooled_feat[target_cols].values.astype(np.float32)
    valid = ~(np.any(np.isnan(oof_lgb), axis=1) | np.any(np.isnan(oof_lstm), axis=1) | np.any(np.isnan(oof_ridge), axis=1) | np.any(np.isnan(oof_ewm), axis=1))
    symbol_list = sorted(pooled_feat["_symbol_"].unique())
    if np.sum(valid) < 5:
        linear_models = []
        meta_scaler = None
    else:
        meta_X_lgb = oof_lgb[valid]
        meta_X_lstm = oof_lstm[valid]
        meta_X_ridge = oof_ridge[valid]
        meta_X_ewm = oof_ewm[valid]
        # OOF base-prediction correlation (per horizon, then mean); high Ridge–LGB/LSTM => Ridge may be redundant
        names = ["lgb", "lstm", "ridge", "ewm"]
        corr_sum = np.zeros((4, 4))
        n_h = 0
        for h in range(horizon):
            arr = np.column_stack([meta_X_lgb[:, h], meta_X_lstm[:, h], meta_X_ridge[:, h], meta_X_ewm[:, h]])
            if np.any(~np.isfinite(arr)):
                continue
            c = np.corrcoef(arr.T)
            if c is not None and c.shape == (4, 4):
                corr_sum += c
                n_h += 1
        if n_h > 0:
            corr_mean = corr_sum / n_h
            oof_corr_df = pd.DataFrame(corr_mean, index=names, columns=names)
            print("OOF base-prediction correlation (mean over horizons):")
            print(oof_corr_df.round(4).to_string())
            r_lgb_ridge = corr_mean[0, 2]
            r_lstm_ridge = corr_mean[1, 2]
            if r_lgb_ridge > 0.9 or r_lstm_ridge > 0.9:
                print("  -> Ridge vs LGB corr = {:.4f}, Ridge vs LSTM corr = {:.4f}. High correlation supports Ridge being redundant (meta may zero it out).".format(r_lgb_ridge, r_lstm_ridge))
        valid_df = pooled_feat.iloc[np.where(valid)[0]]
        meta_ctx = valid_df[["month_sin", "month_cos", "vix_velocity", "rolling_vol"]].values.astype(np.float32)
        sym_oh = np.array([(np.array(symbol_list) == s).astype(np.float32) for s in valid_df["_symbol_"].values])
        meta_y = y[valid]
        X_meta_list = []
        for h in range(horizon):
            disagreement_h = np.std(np.column_stack([meta_X_lgb[:, h], meta_X_lstm[:, h], meta_X_ridge[:, h], meta_X_ewm[:, h]]), axis=1, keepdims=True).astype(np.float32)
            X_meta_list.append(np.hstack([meta_X_lgb[:, h : h + 1], meta_X_lstm[:, h : h + 1], meta_X_ridge[:, h : h + 1], meta_X_ewm[:, h : h + 1], disagreement_h, meta_ctx, sym_oh]))
        X_meta_all = np.vstack(X_meta_list)
        meta_scaler = StandardScaler()
        meta_scaler.fit(X_meta_all)
        linear_models = []
        for h in range(horizon):
            disagreement_h = np.std(np.column_stack([meta_X_lgb[:, h], meta_X_lstm[:, h], meta_X_ridge[:, h], meta_X_ewm[:, h]]), axis=1, keepdims=True).astype(np.float32)
            X_meta = np.hstack([meta_X_lgb[:, h : h + 1], meta_X_lstm[:, h : h + 1], meta_X_ridge[:, h : h + 1], meta_X_ewm[:, h : h + 1], disagreement_h, meta_ctx, sym_oh])
            X_meta_s = meta_scaler.transform(X_meta)
            ridge_meta = Ridge(alpha=RIDGE_META_ALPHA, positive=True, random_state=42)
            ridge_meta.fit(X_meta_s, meta_y[:, h])
            linear_models.append(ridge_meta)
    # Final base training on full training set (per-symbol scalers)
    y_full = pooled_feat[target_cols].values.astype(np.float32)
    scaler_xgb_by_sym = {}
    scaler_lstm_by_sym = {}
    scaler_ridge_by_sym = {}
    for sym in pooled_feat["_symbol_"].unique():
        m = pooled_feat["_symbol_"] == sym
        scaler_xgb_by_sym[sym] = StandardScaler().fit(pooled_feat.loc[m, feature_cols_xgb].values.astype(np.float32))
        scaler_lstm_by_sym[sym] = StandardScaler().fit(pooled_feat.loc[m, feature_cols_lstm].values.astype(np.float32))
        scaler_ridge_by_sym[sym] = StandardScaler().fit(pooled_feat.loc[m, feature_cols_ridge].values.astype(np.float64))
    X_xgb_full_s = np.empty((len(pooled_feat), len(feature_cols_xgb)), dtype=np.float32)
    X_lstm_full_s = np.empty((len(pooled_feat), len(feature_cols_lstm)), dtype=np.float32)
    X_ridge_full_s = np.empty((len(pooled_feat), len(feature_cols_ridge)), dtype=np.float64)
    for sym in pooled_feat["_symbol_"].unique():
        m = pooled_feat["_symbol_"] == sym
        X_xgb_full_s[m] = scaler_xgb_by_sym[sym].transform(pooled_feat.loc[m, feature_cols_xgb].values.astype(np.float32))
        X_lstm_full_s[m] = scaler_lstm_by_sym[sym].transform(pooled_feat.loc[m, feature_cols_lstm].values.astype(np.float32))
        X_ridge_full_s[m] = scaler_ridge_by_sym[sym].transform(pooled_feat.loc[m, feature_cols_ridge].values.astype(np.float64))
    lgb_multi = MultiOutputRegressor(lgb.LGBMRegressor(**LGB_PARAMS))
    lgb_multi.fit(X_xgb_full_s, y_full)
    X_seq_list, y_seq_list = [], []
    for sym in pooled_feat["_symbol_"].unique():
        sym_mask = pooled_feat["_symbol_"] == sym
        X_s = X_lstm_full_s[sym_mask]
        y_s = y_full[sym_mask]
        X_seq, y_seq = build_sequences(X_s, y_s, SEQ_LEN)
        if X_seq is not None and len(X_seq) >= 10:
            X_seq_list.append(X_seq)
            y_seq_list.append(y_seq)
    lstm_model = train_lstm(np.concatenate(X_seq_list, axis=0), np.concatenate(y_seq_list, axis=0), horizon, epochs=LSTM_EPOCHS_FINAL, use_early_stopping=True) if X_seq_list else None
    ridge_multi = MultiOutputRegressor(Ridge(alpha=RIDGE_ALPHA, random_state=42))
    ridge_multi.fit(X_ridge_full_s, y_full.astype(np.float64))
    return {
        "scaler_xgb_by_sym": scaler_xgb_by_sym, "scaler_lstm_by_sym": scaler_lstm_by_sym, "scaler_ridge_by_sym": scaler_ridge_by_sym,
        "meta_scaler": meta_scaler,
        "lgb_multi": lgb_multi, "lstm_model": lstm_model, "ridge_multi": ridge_multi, "linear_models": linear_models,
        "feature_cols_lstm": feature_cols_lstm, "feature_cols_xgb": feature_cols_xgb, "feature_cols_ridge": feature_cols_ridge,
        "target_cols": target_cols, "EWM_SPAN": EWM_SPAN, "meta_symbol_list": symbol_list,
    }


def predict_stack_ridge_global(context_df: pd.DataFrame, horizon: int, global_stack: dict) -> list:
    """Predict 21 price steps using pre-trained stack (LightGBM + LSTM + Ridge + EWM, Ridge meta on 4 bases + disagreement + month_sin/month_cos/vix_velocity/rolling_vol + symbol one-hot)."""
    if global_stack is None or not global_stack.get("linear_models") or global_stack.get("meta_scaler") is None:
        return []
    try:
        feat_df, feature_cols_lstm, feature_cols_xgb, _ = build_feature_df(context_df)
    except Exception:
        return []
    if len(feat_df) < SEQ_LEN + 1:
        return []
    feature_cols_ridge = global_stack.get("feature_cols_ridge", feature_cols_lstm + feature_cols_xgb)
    sym_d = global_stack.get("scaler_xgb_by_sym", {})
    symbol = context_df["symbol"].iloc[0] if "symbol" in context_df.columns and len(context_df) else (list(sym_d.keys())[0] if sym_d else None)
    if symbol is None:
        return []
    scaler_xgb = sym_d.get(symbol, list(sym_d.values())[0]) if sym_d else None
    scaler_lstm = global_stack.get("scaler_lstm_by_sym", {}).get(symbol, list(global_stack.get("scaler_lstm_by_sym", {}).values())[0]) if global_stack.get("scaler_lstm_by_sym") else None
    scaler_ridge = global_stack.get("scaler_ridge_by_sym", {}).get(symbol, list(global_stack.get("scaler_ridge_by_sym", {}).values())[0]) if global_stack.get("scaler_ridge_by_sym") else None
    if scaler_xgb is None or scaler_lstm is None or scaler_ridge is None:
        return []
    lgb_multi = global_stack["lgb_multi"]
    lstm_model = global_stack["lstm_model"]
    ridge_multi = global_stack["ridge_multi"]
    linear_models = global_stack["linear_models"]
    EWM_SPAN = global_stack.get("EWM_SPAN", 20)
    X_xgb_s = scaler_xgb.transform(feat_df[feature_cols_xgb].values.astype(np.float32))
    X_lstm_s = scaler_lstm.transform(feat_df[feature_cols_lstm].values.astype(np.float32))
    X_ridge_s = scaler_ridge.transform(feat_df[feature_cols_ridge].values.astype(np.float64))
    last_idx = len(feat_df) - 1
    last_row_xgb = X_xgb_s[last_idx : last_idx + 1]
    last_row_ridge = X_ridge_s[last_idx : last_idx + 1]
    xgb_21 = lgb_multi.predict(last_row_xgb).ravel()
    ridge_21 = ridge_multi.predict(last_row_ridge).ravel()
    ret_series = feat_df["return"].values.astype(np.float64)
    if len(ret_series) >= EWM_SPAN:
        ewm_val = float(pd.Series(ret_series).ewm(span=EWM_SPAN).mean().iloc[-1])
        ewm_21 = np.full(horizon, ewm_val, dtype=np.float32)
    else:
        ewm_21 = np.full(horizon, float(np.nanmean(ret_series)) if len(ret_series) else 0.0, dtype=np.float32)
    if lstm_model is not None and last_idx >= SEQ_LEN:
        seq = X_lstm_s[last_idx - SEQ_LEN : last_idx]
        lstm_21 = lstm_model.predict(seq.reshape(1, SEQ_LEN, -1), verbose=0).ravel()
    else:
        lstm_21 = xgb_21
    vv = feat_df["vix_velocity"].iloc[-1]
    vix_vel = np.nan_to_num(float(vv), nan=0.0)
    roll_vol = float(feat_df["rolling_vol"].iloc[-1]) if "rolling_vol" in feat_df.columns else 0.0
    ctx_last = np.concatenate([feat_df[["month_sin", "month_cos"]].iloc[-1].values.astype(np.float32), [np.float32(vix_vel), np.float32(roll_vol)]])
    symbol_list = global_stack.get("meta_symbol_list", [])
    sym_oh = (np.array(symbol_list) == symbol).astype(np.float32) if symbol_list else np.zeros(0, dtype=np.float32)
    meta_vec = np.array([np.concatenate([np.array([xgb_21[h], lstm_21[h], ridge_21[h], ewm_21[h], np.std([xgb_21[h], lstm_21[h], ridge_21[h], ewm_21[h]])], dtype=np.float32), ctx_last, sym_oh]) for h in range(horizon)])
    meta_vec_s = global_stack["meta_scaler"].transform(meta_vec)
    final_returns = np.array([linear_models[h].predict(meta_vec_s[h : h + 1])[0] for h in range(horizon)])
    p0 = float(context_df["close"].iloc[-1])
    prices = p0 * np.cumprod(np.concatenate([[1.0], 1.0 + final_returns]))[1:]
    return [float(p) for p in prices[:horizon]]


In [27]:
stacked = load_pool_data(with_vix=True, with_volume=True)
symbol_start = pd.to_datetime(stacked["timestamp"]).min().strftime("%Y-%m-%d")
fear_greed_df = fetch_cnn_fear_greed_index(start_date=symbol_start)
if not fear_greed_df.empty:
    stacked["date"] = pd.to_datetime(stacked["timestamp"]).dt.normalize()
    fear_greed_df["date"] = pd.to_datetime(fear_greed_df["timestamp"]).dt.normalize()
    stacked = stacked.merge(fear_greed_df[["date", "fear_greed"]], on="date", how="left")
    stacked["fear_greed"] = stacked["fear_greed"].ffill().bfill()
    stacked = stacked.drop(columns=["date"])
print(stacked.groupby("symbol").size())
stacked.head(10)

symbol
AAPL     1256
AMZN     1256
GOOGL    1256
JNJ      1256
JPM      1256
MSFT     1256
NVDA     1256
SPY      1256
WMT      1256
XOM      1256
dtype: int64


,timestamp,symbol,close,volume,vix,fear_greed
0,2021-03-09,AAPL,121.089996,129525800,24.030001,43.360000
1,2021-03-10,AAPL,119.980003,111943300,22.559999,45.560000
2,2021-03-11,AAPL,121.959999,103026500,21.910000,50.480000
3,2021-03-12,AAPL,121.029999,88105100,20.690001,53.720000
4,2021-03-15,AAPL,123.989998,92403800,20.030001,56.520000
5,2021-03-16,AAPL,125.570000,115227900,19.790001,54.800000
6,2021-03-17,AAPL,124.760002,111932600,19.230000,57.866667
7,2021-03-18,AAPL,120.529999,121229700,21.580000,52.333333
8,2021-03-19,AAPL,119.989998,185549500,20.950001,50.833333
9,2021-03-22,AAPL,123.389999,111912300,18.879999,50.700000


In [28]:
stacked.describe()

,timestamp,close,volume,vix,fear_greed
count,12560,12560.000000,1.256000e+04,12560.000000,12560.000000
mean,2023-09-05 01:53:30,195.188046,7.072414e+07,19.141290,47.486396
min,2021-03-09 00:00:00,11.227000,2.316700e+06,11.860000,2.900000
25%,2022-06-05 06:00:00,109.519999,1.489950e+07,15.487500,32.992857
50%,2023-09-05 12:00:00,158.820000,2.864700e+07,17.889999,47.900000
75%,2024-12-03 06:00:00,235.992504,6.280765e+07,21.580000,62.948810
max,2026-03-09 00:00:00,695.489990,1.543911e+09,52.330002,82.971429
std,NaN,138.462988,1.245263e+08,5.197167,18.348596


In [29]:
# Train once on pooled data (all assets, only rows before 60-day test window)
global_stack = train_global_stack_ridge(stacked, FORECAST_HORIZON)
pooled_train = build_pooled_train_stack(stacked, TEST_SIZE, MIN_TRAIN_STACK)
print("Global stack trained on", len(pooled_train), "pooled train rows.")

c:\capstone_project_unfc\env\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
c:\capstone_project_unfc\env\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
c:\capstone_project_unfc\env\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
c:\capstone_project_unfc\env\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`

OOF base-prediction correlation (mean over horizons):
          lgb    lstm   ridge     ewm
lgb    1.0000  0.0069  0.3122 -0.0879
lstm   0.0069  1.0000 -0.0135 -0.0235
ridge  0.3122 -0.0135  1.0000 -0.3626
ewm   -0.0879 -0.0235 -0.3626  1.0000


c:\capstone_project_unfc\env\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Global stack trained on 11960 pooled train rows.


In [30]:
# Backtest: 60-day test window, 21-day horizon, step 7 days; predict only (global models trained once above)
model_name = "stack_ridge_meta"
all_preds = []
for sym in TICKERS:
    grp = stacked[stacked["symbol"] == sym].copy()
    if grp.empty:
        continue
    grp = grp.sort_values("timestamp").reset_index(drop=True)
    prices = grp.set_index("timestamp")["close"].astype(float).dropna()
    n = len(prices)
    if n < TEST_SIZE + MIN_TRAIN_STACK:
        continue
    split_idx = n - TEST_SIZE
    test_index = prices.index[split_idx:]
    test_values = prices.values[split_idx:]
    preds = []
    window_ix = 0
    start = 0
    while start + FORECAST_HORIZON <= TEST_SIZE:
        context_cols = ["timestamp", "close", "vix", "symbol"] + [c for c in ["volume", "fear_greed"] if c in grp.columns]
        context_df = grp.iloc[: split_idx + start][context_cols].copy()
        if len(context_df) < MIN_TRAIN_STACK:
            start += ROLLING_STEP
            continue
        price_list = predict_stack_ridge_global(context_df, FORECAST_HORIZON, global_stack)
        if not price_list or len(price_list) < FORECAST_HORIZON:
            start += ROLLING_STEP
            window_ix += 1
            continue
        for h in range(FORECAST_HORIZON):
            idx = start + h
            ts = test_index[idx]
            y_true = float(test_values[idx])
            y_pred = float(price_list[h])
            preds.append({"timestamp": ts, "y_true": y_true, "y_pred": y_pred, "window_ix": window_ix})
        window_ix += 1
        start += ROLLING_STEP
    if preds:
        pred_df = pd.DataFrame(preds)
        pred_df["symbol"] = sym
        all_preds.append(pred_df)

pred_stack = pd.concat(all_preds, ignore_index=True) if all_preds else pd.DataFrame(
    columns=["timestamp", "y_true", "y_pred", "window_ix", "symbol"]
)
print(pred_stack.groupby("symbol").size() if not pred_stack.empty else "No predictions.")
pred_stack.head()


symbol
AAPL     126
AMZN     126
GOOGL    126
JNJ      126
JPM      126
MSFT     126
NVDA     126
SPY      126
WMT      126
XOM      126
dtype: int64


,timestamp,y_true,y_pred,window_ix,symbol
0,2025-12-10,278.779999,277.319795,0,AAPL
1,2025-12-11,278.029999,277.483173,0,AAPL
2,2025-12-12,278.279999,277.557302,0,AAPL
3,2025-12-15,274.109985,277.644388,0,AAPL
4,2025-12-16,274.609985,277.780288,0,AAPL


In [31]:
# Global stack is trained in the cell above. Run that cell before the backtest.

In [32]:
metrics_rows = []
for sym in pred_stack["symbol"].unique():
    sub = pred_stack[pred_stack["symbol"] == sym]
    m = compute_metrics_averaged_over_windows(sub)
    metrics_rows.append({"model": model_name, "symbol": sym, **m})
m_overall = compute_metrics_averaged_over_windows(pred_stack)
metrics_rows.append({"model": model_name, "symbol": "overall", **m_overall})

metrics_df = pd.DataFrame(metrics_rows)
print(metrics_df.to_string())
metrics_to_parquet(metrics_rows, ARTIFACTS_DIR / "metrics_stack_ridge_meta_pool.parquet")
print("Saved:", ARTIFACTS_DIR / "metrics_stack_ridge_meta_pool.parquet")

               model   symbol        MAE       RMSE    MAPE_%
0   stack_ridge_meta     AAPL  11.045613  13.217135  4.200959
1   stack_ridge_meta     MSFT  26.459714  30.728290  6.181302
2   stack_ridge_meta    GOOGL  12.991630  15.002764  4.087313
3   stack_ridge_meta     AMZN  13.284739  15.668923  6.107703
4   stack_ridge_meta      JPM  13.653306  15.146534  4.357092
5   stack_ridge_meta      JNJ  11.351631  12.585428  4.947856
6   stack_ridge_meta      WMT   4.674945   5.390931  3.808668
7   stack_ridge_meta      SPY   6.166792   7.301567  0.898121
8   stack_ridge_meta      XOM   7.490143   8.814798  5.352295
9   stack_ridge_meta     NVDA   6.568427   8.062561  3.591691
10  stack_ridge_meta  overall  11.368694  15.544826  4.353300
Saved: C:\capstone_project_unfc\model\experiments-pool\artifacts\metrics_stack_ridge_meta_pool.parquet


In [33]:
# Ridge meta-learner coefficients (one per horizon; features: lgb, lstm, ridge, ewm, disagreement, month_sin, month_cos, vix_velocity, rolling_vol, symbol one-hot)
sym = TICKERS[0]
grp = stacked[stacked["symbol"] == sym].sort_values("timestamp").reset_index(drop=True)
n = len(grp)
if n >= TEST_SIZE + MIN_TRAIN_STACK:
    linear_models = (global_stack or {}).get("linear_models", [])
    symbol_list = (global_stack or {}).get("meta_symbol_list", [])
    meta_feature_names = ["coef_lgb", "coef_lstm", "coef_ridge", "coef_ewm", "coef_disagreement", "coef_month_sin", "coef_month_cos", "coef_vix_velocity", "coef_rolling_vol"] + [f"coef_sym_{s}" for s in symbol_list]
    if linear_models:
        rows = []
        for h, ridge_meta in enumerate(linear_models):
            coef = ridge_meta.coef_
            rows.append({"step": h + 1, **dict(zip(meta_feature_names, coef))})
        meta_df = pd.DataFrame(rows)
        print("Ridge meta-learner coefficients (lgb, lstm, ridge, ewm, disagreement, month_sin, month_cos, vix_velocity, rolling_vol, symbol one-hot)")
        print(meta_df.to_string(index=False))
    else:
        print("Could not obtain linear models.")
else:
    print("Not enough data for meta-learner coefficients.")

Ridge meta-learner coefficients (lgb, lstm, ridge, ewm, disagreement, month_sin, month_cos, vix_velocity, rolling_vol, symbol one-hot)
 step  coef_lgb  coef_lstm  coef_ridge  coef_ewm  coef_disagreement  coef_month_sin  coef_month_cos  coef_vix_velocity  coef_rolling_vol  coef_sym_AAPL  coef_sym_AMZN  coef_sym_GOOGL  coef_sym_JNJ  coef_sym_JPM  coef_sym_MSFT  coef_sym_NVDA  coef_sym_SPY  coef_sym_WMT  coef_sym_XOM
    1  0.001025   0.000400         0.0  0.000000           0.000000        0.000020        0.000015           0.000311          0.000000       0.000000       0.000000        0.000042           0.0      0.000075       0.000017       0.000547      0.000000      0.000131      0.000070
    2  0.000000   0.000105         0.0  0.000000           0.000000        0.000000        0.000043           0.000000          0.000187       0.000000       0.000000        0.000003           0.0      0.000053       0.000000       0.000464      0.000000      0.000089      0.000041
    3  0.000000 